# Paper 17 · LoRA

**Citation:** Edward J. Hu et al., “LoRA: Low-Rank Adaptation of Large Language Models” (2021).

**Paper:** https://arxiv.org/abs/2106.09685

> **Scale gap:** We adapt a frozen linear transformation whose true task shift is low-rank. This isolates the mechanism without an LLM.

## Mathematical Framework

Before reproducing the paper experimentally, work through:

- [Math 01 · Linear Algebra & Geometry](../../math/01_linear_algebra_geometry.ipynb)
- [Math 09 · PCA, SVD & Kernels](../../math/09_pca_svd_kernels.ipynb)

Your explanation should connect the paper's empirical claim to its **mathematical objective, representation, assumptions, and optimization/statistical argument**.

## Before you read
1. What is the parameter saving from rank r?
2. When would a low-rank update fail?
3. Why is keeping the base frozen operationally useful?

## Central claim
Many adaptation updates can be represented effectively with low-rank trainable matrices while base model weights remain frozen.

## Construct a task with a low-rank shift

In [ ]:
import torch, pandas as pd, matplotlib.pyplot as plt
from torch import nn
torch.manual_seed(0)
din=dout=32
W0=torch.randn(dout,din)/din**.5
Atrue=torch.randn(2,din)*.5; Btrue=torch.randn(dout,2)*.5
Wstar=W0+Btrue@Atrue
X=torch.randn(3000,din); Y=X@Wstar.T
Xtr,Xte=X[:2400],X[2400:]; Ytr,Yte=Y[:2400],Y[2400:]

## LoRA module
TODO: derive the parameter count before running the sweep.

In [ ]:
class LoRA(nn.Module):
    def __init__(self,r):
        super().__init__(); self.r=r
        self.A=nn.Parameter(torch.randn(r,din)*.01)
        self.B=nn.Parameter(torch.zeros(dout,r))
    def forward(self,x):
        return x@W0.T + (x@self.A.T)@self.B.T

def fit_rank(r,steps=300):
    m=LoRA(r); opt=torch.optim.Adam(m.parameters(),lr=.03)
    for _ in range(steps):
        pred=m(Xtr); loss=((pred-Ytr)**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): mse=((m(Xte)-Yte)**2).mean().item()
    return mse,sum(p.numel() for p in m.parameters())

## Rank sweep

In [ ]:
rows=[]
for r in [1,2,4,8,16]:
    mse,n=fit_rank(r)
    rows.append({"rank":r,"trainable_params":n,"test_MSE":mse})
df=pd.DataFrame(rows); display(df)
plt.plot(df["rank"],df["test_MSE"],marker="o"); plt.yscale("log"); plt.xlabel("LoRA rank"); plt.ylabel("test MSE"); plt.show()
print("full matrix params:",din*dout)

### Ablation
Regenerate `Wstar-W0` as a full-rank random matrix. Repeat the rank sweep. When does low-rank adaptation stop being sufficient?

## Ablation table

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
1. What problem existed before this work?
2. What was actually new?
3. What evidence did this notebook reproduce?
4. What does the scale gap prevent you from claiming?
5. Which contribution remains important today?
6. What would you test next?